In [0]:
CREATE DATABASE IF NOT EXISTS nyc_taxi.bronze;
CREATE DATABASE IF NOT EXISTS nyc_taxi.silver;
CREATE DATABASE IF NOT EXISTS nyc_taxi.gold;

## **1. BRONZE LAYER SCHEMA**

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze.payment (
    payment_type_code STRING,
    payment_type STRING,
    file_name STRING,
    created_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/bronze/payment'
TBLPROPERTIES('delta.enableChangeDataFeed' = 'true');

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze.zone (
    LocationID STRING,
    Borough STRING,
    Zone STRING,
    service_zone STRING,
    file_name STRING,
    created_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/bronze/zone'
TBLPROPERTIES('delta.enableChangeDataFeed' = 'true');

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze.type(
    trip_type STRING,
    description STRING,
    file_name STRING,
    created_on TIMESTAMP
) USING DELTA LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/bronze/type'
TBLPROPERTIES('delta.enableChangeDataFeed' = 'true');

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.bronze.green_trip (
    VendorID INT,
    lpep_pickup_datetime TIMESTAMP_NTZ,
    lpep_dropoff_datetime TIMESTAMP_NTZ,
    store_and_fwd_flag STRING,
    RatecodeID BIGINT,
    PULocationID INT,
    DOLocationID INT,
    passenger_count BIGINT,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    ehail_fee DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    payment_type BIGINT,
    trip_type BIGINT,
    congestion_surcharge DOUBLE,
    year INT,
    month INT,
    file_name STRING,
    created_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/bronze/green_taxi'
PARTITIONED BY (year, month)
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

## **2. SILVER LAYER SCHEMA**

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.silver.type (
    trip_type_id INT,
    trip_type STRING,
    created_on TIMESTAMP,
    modified_on TIMESTAMP NOT NULL
)
USING DELTA
LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/silver/type'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');


In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.silver.zone (
    zone_id INT,
    borough STRING,
    zone1 STRING,
    zone2 STRING,
    service_zone STRING,
    created_on TIMESTAMP,
    modified_on TIMESTAMP
)
USING DELTA
LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/silver/zone'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.silver.payment(
    payment_type_id INT,
    payment_type STRING,
    created_on TIMESTAMP,
    modified_on TIMESTAMP
) USING DELTA 
LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/silver/payment'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.silver.green_trip (
    deterministic_hash_key STRING,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    pickup_zone_id INT,
    dropoff_zone_id INT,
    payment_id INT,
    trip_type_id INT,
    passenger_count INT,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    total_amount DOUBLE,
    year INT,
    month INT,
    trip_duration DOUBLE,
    average_speed DOUBLE,
    extra_charge DOUBLE,
    created_on TIMESTAMP,
    modified_on TIMESTAMP
)
USING DELTA LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/silver/green_taxi'
PARTITIONED BY (year, month)
TBLPROPERTIES('delta.enableChangeDataFeed' = 'true');

## **3. GOLD LAYER SCHEMA**

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.gold.dim_payment(
    payment_type_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    payment_type_id INTEGER,
    payment_type STRING
)
USING DELTA LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/gold/payment';

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.gold.dim_type(
    trip_type_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    trip_type_id INTEGER,
    trip_type STRING
)
USING DELTA LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/gold/type';

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.gold.dim_zone (
    zone_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    zone_id INTEGER,
    borough STRING,
    service_zone STRING,
    zone1 STRING,
    zone2 STRING
)
USING DELTA LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/gold/zone';

In [0]:
CREATE TABLE IF NOT EXISTS nyc_taxi.gold.fact_trips (
    trip_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    trip_id STRING,
    pickup_zone_sk LONG,
    dropoff_zone_sk LONG,
    trip_type_sk LONG,
    payment_type_sk LONG,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    passenger_count INT,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    total_amount DOUBLE,
    extra_charge DOUBLE,
    trip_duration DOUBLE,
    average_speed DOUBLE,
    year INT,
    month INT
)
USING DELTA
LOCATION 'abfss://nyc-taxi-data@gen2nyctaxi.dfs.core.windows.net/gold/fact_trips'
PARTITIONED BY (year, month)